## HW 4

In this assignment, we will use ALS to predict ratings of Amazon Videos
### Assignment submission group
- Group member 1: Chakat Srisuvanutna (cs635@drexel.edu)
- Group member 2: Alvin Boakai (asb424@drexel.edu)

In [14]:
import os
os.environ.pop("SPARK_HOME", None)
os.environ.pop("JAVA_HOME", None)

!pip install -q pyspark

from google.colab import drive
drive.mount('/content/gdrive')

FILE_NAME = 'ratings_Amazon_Instant_Video.csv'
FILE_PATH = f'/content/gdrive/MyDrive/Colab Notebooks/HW4/{FILE_NAME}'

import os
if os.path.exists(FILE_PATH):
    print(f"✅ Found: {FILE_PATH}")
else:
    print(f"❌ Not found: {FILE_PATH}")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
✅ Found: /content/gdrive/MyDrive/Colab Notebooks/HW4/ratings_Amazon_Instant_Video.csv


In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("HW4").getOrCreate()
spark

In [16]:
from google.colab import drive
drive.mount('/content/gdrive')

APP_NAME = "HW4"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [17]:
spark = SparkSession.builder.appName(APP_NAME).getOrCreate()

In [18]:
spark

1.Load the Amazon Video Ratings dataset and note that there are no column names in this dataset.

In [13]:
df = spark.read.csv(FILE_PATH, header=False, inferSchema=True)
df.show(5)
print(f"Rows: {df.count()}")
print(f"Cols: {len(df.columns)}")

+--------------+----------+---+----------+
|           _c0|       _c1|_c2|       _c3|
+--------------+----------+---+----------+
|A1EE2E3N7PW666|B000GFDAUG|5.0|1202256000|
| AGZ8SM1BGK3CK|B000GFDAUG|5.0|1198195200|
|A2VHZ21245KBT7|B000GIOPK2|4.0|1215388800|
| ACX8YW2D5EGP6|B000GIOPK2|4.0|1185840000|
| A9RNMO9MUSMTJ|B000GIOPK2|2.0|1281052800|
+--------------+----------+---+----------+
only showing top 5 rows
Rows: 583933
Cols: 4


2. Take a sample of 5% of the data and assign that sample to a new variable. You may use the sample function provided in PySpark. Since we are only trying out a prototype, sampling will ensure our code will run much faster.

In [19]:
df_sample = df.sample(fraction=0.05, seed=42)

print(f"Original rows: {df.count()}")
print(f"Sampled rows:  {df_sample.count()}")
df_sample.show(5)

Original rows: 583933
Sampled rows:  29254
+--------------+----------+---+----------+
|           _c0|       _c1|_c2|       _c3|
+--------------+----------+---+----------+
| AVE3EF44DFS0C|B000GIOPK2|5.0|1190937600|
|A3QW44Q3C856NR|B000GK6NFK|5.0|1290470400|
| ABUZCXVWZLHC9|B000GK6NFK|3.0|1223683200|
|A1FTD75LGWVKPH|B000GK6NFK|5.0|1361404800|
|A3H8F10BFSZ14U|B000GK6NFK|5.0|1374019200|
+--------------+----------+---+----------+
only showing top 5 rows


3. Rename the columns to: user_id, product_id, rating, time_stamp. Assign the dataframe to a new variable.

In [20]:
df_renamed = df_sample.toDF("user_id", "product_id", "rating", "time_stamp")
df_renamed.show(5)
df_renamed.printSchema()

+--------------+----------+------+----------+
|       user_id|product_id|rating|time_stamp|
+--------------+----------+------+----------+
| AVE3EF44DFS0C|B000GIOPK2|   5.0|1190937600|
|A3QW44Q3C856NR|B000GK6NFK|   5.0|1290470400|
| ABUZCXVWZLHC9|B000GK6NFK|   3.0|1223683200|
|A1FTD75LGWVKPH|B000GK6NFK|   5.0|1361404800|
|A3H8F10BFSZ14U|B000GK6NFK|   5.0|1374019200|
+--------------+----------+------+----------+
only showing top 5 rows
root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- time_stamp: integer (nullable = true)



4. Since we have a time stamp column, this may mean that users reviewed the same video more than once at different times. Decide on the best way to ensure there is only one review per product-user combination and aggregate the data accordingly. Explain in a comment why you chose this aggregation method.

In [21]:
from pyspark.sql.functions import count

# Not found user-product pairs more than one review
duplicates = (
    df_renamed
    .groupBy("user_id", "product_id")
    .agg(count("*").alias("review_count"))
    .filter("review_count > 1")
)

print(f"Duplicate user-product pairs: {duplicates.count()}")
duplicates.show(10)

Duplicate user-product pairs: 0
+-------+----------+------------+
|user_id|product_id|review_count|
+-------+----------+------------+
+-------+----------+------------+



In [22]:
# Use mean rating because a user may rate the same product differently over time.
# Taking the average fairly represents their overall sentiment
# rather than keeping only the first or last review.
from pyspark.sql.functions import mean, round as spark_round

df_agg = (
    df_renamed
    .groupBy("user_id", "product_id")
    .agg(spark_round(mean("rating"), 1).alias("rating"))
)

df_agg.show(20)
print(f"Rows before aggregation: {df_renamed.count()}")
print(f"Rows after aggregation:  {df_agg.count()}")

+--------------+----------+------+
|       user_id|product_id|rating|
+--------------+----------+------+
|A3LR0NNIJ05TPU|B000OGTRC2|   5.0|
|A2ZCFJ6EOMN0Y4|B000S9FEFY|   5.0|
| A3F3ZKYC8WVYG|B000U6I0B0|   5.0|
|A26RMIL4VZZP28|B000U6YXPM|   5.0|
|A2SI2TNVOBJWP6|B000VU2SW2|   4.0|
|A3OZHGJME0FEVY|B000VU4GW2|   4.0|
| A54VQWYT66UIA|B000WDS04I|   5.0|
|A3IJLOBZSRLMMO|B0012QRPU4|   5.0|
|A1U5XKFJJUQES0|B001DCX6BO|   5.0|
| AFULMWJIG6N2J|B001G8MP9Y|   5.0|
|A3LWJXY1UPA6N5|B001ODQA4C|   5.0|
|A1IPN8WIWC01H3|B0021Y8RW6|   4.0|
|A2OISMXEBDGJS0|B002AL4A4E|   5.0|
|A202NBIARN2JHX|B002L4BQ42|   5.0|
|A1LRMSKYVL7QL2|B002NWNTL0|   5.0|
|A13KU488TJE4PV|B002Y0TGOU|   4.0|
|A1F4ZXH3EYFTEE|B003075T38|   1.0|
| AACVOLN9G659Z|B00366JI9O|   5.0|
|A2GN9WVQO5CX65|B0039HW0HM|   5.0|
|A13WLS3G3OM50C|B003B63FKW|   5.0|
+--------------+----------+------+
only showing top 20 rows
Rows before aggregation: 29254
Rows after aggregation:  29254


5. Add an integer id column for both user and product.

In [23]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

df_ids = (
    df_agg
    .withColumn("user_int_id",
        dense_rank().over(Window.orderBy("user_id")))
    .withColumn("product_int_id",
        dense_rank().over(Window.orderBy("product_id")))
)

df_ids.show(10)
df_ids.printSchema()

+--------------+----------+------+-----------+--------------+
|       user_id|product_id|rating|user_int_id|product_int_id|
+--------------+----------+------+-----------+--------------+
| AVE3EF44DFS0C|B000GIOPK2|   5.0|      27373|             1|
|A2OENV0Q6XM7RV|B000GK6NFK|   5.0|      12721|             2|
|A1FTD75LGWVKPH|B000GK6NFK|   5.0|       3324|             2|
|A3H8F10BFSZ14U|B000GK6NFK|   5.0|      18661|             2|
|A3QW44Q3C856NR|B000GK6NFK|   5.0|      20629|             2|
| ABUZCXVWZLHC9|B000GK6NFK|   3.0|      23353|             2|
| A2GD669PGTMJZ|B000GOTJGG|   1.0|      11050|             3|
|A1CIZ90FFOZTLM|B000H00VBQ|   5.0|       2600|             4|
| A3O516U8OVPRX|B000H00VBQ|   2.0|      20050|             4|
|A1JR35ENU57PKK|B000H0X79O|   5.0|       4179|             5|
+--------------+----------+------+-----------+--------------+
only showing top 10 rows
root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- rating: double (

In [24]:
# Show only users who reviewed more than one product.
from pyspark.sql.functions import count

df_ids.groupBy("user_id") \
    .agg(count("product_id").alias("product_count")) \
    .filter("product_count > 1") \
    .join(df_ids, on="user_id") \
    .orderBy("user_id", "product_id") \
    .show(20)

+--------------+-------------+----------+------+-----------+--------------+
|       user_id|product_count|product_id|rating|user_int_id|product_int_id|
+--------------+-------------+----------+------+-----------+--------------+
|A105188E1HFWRX|            2|B007JF83YY|   4.0|         54|          2815|
|A105188E1HFWRX|            2|B00IMYQMH6|   5.0|         54|          5765|
|A10ALOMJECNL2H|            2|B00ATLJYL6|   5.0|         86|          3914|
|A10ALOMJECNL2H|            2|B00F49E8O6|   5.0|         86|          5067|
|A10D3I4NV0H6E5|            2|B001NWFLHQ|   5.0|        102|           565|
|A10D3I4NV0H6E5|            2|B00BGAXI3Y|   5.0|        102|          4128|
|A10KK981NCHT9Y|            2|B004J3CO5S|   4.0|        156|          1789|
|A10KK981NCHT9Y|            2|B00HMVFAIM|   1.0|        156|          5536|
|A10KWK15A6273V|            2|B00GM5TUJO|   5.0|        159|          5364|
|A10KWK15A6273V|            2|B00KT19HJA|   4.0|        159|          6044|
|A10UGBZ4C8D

6. Split the data into train and test with 20% in test

In [25]:
train_df, test_df = df_ids.randomSplit([0.8, 0.2], seed=42)

print(f"Train size: {train_df.count()}")
print(f"Test size:  {test_df.count()}")

Train size: 23439
Test size:  5815


7. Create an ALS model to predict the ratings. set the setMaxIter value to 3.
Fit the model to the training data and transform on the test data. Is the data explicit or implicit? Set the parameter accordingly.

In [26]:
# Data is EXPLICIT — users gave direct ratings (1-5 stars)
# so implicitPrefs = False (default)

from pyspark.ml.recommendation import ALS

als = ALS(
    maxIter=3,
    userCol="user_int_id",
    itemCol="product_int_id",
    ratingCol="rating",
    implicitPrefs=False,       # explicit ratings
    coldStartStrategy="drop",  # drop NaN predictions for unseen users/items
    seed=42
)

als_model = als.fit(train_df)
predictions = als_model.transform(test_df)

8. Print 20 rows from the prediction table

In [29]:
predictions.select("user_id", "product_id", "rating", "prediction").show(20)

+--------------+----------+------+-----------+
|       user_id|product_id|rating| prediction|
+--------------+----------+------+-----------+
|A10UGBZ4C8DBOY|B00I3MNGCG|   4.0|  1.0893525|
|A10WV4S63AE44R|B0055EQ30M|   5.0| 0.10219669|
|A10ZQTGUHHF670|B00F2CE91W|   5.0|   5.169137|
|A110GVO80Z9NQ5|B00BPDFXDA|   5.0|  2.5610359|
|A11V0MR0IYVK9X|B005OV0LJA|   5.0|  6.9512453|
|A12IRGQLFE4EBA|B00B19GYCW|   4.0|-0.25938576|
|A12X3J7IITW1J6|B0087BJXM0|   5.0| -1.5449278|
|A14GK0E64J0WAS|B00CO0KZK4|   4.0|  0.6176987|
|A16JX9D1DGPFCP|B0084YBGMK|   5.0| 0.52488834|
|A182HG8KEIZ5YN|B00I3MPDP4|   5.0| 0.03663175|
|A18HE80910BTZI|B00DTOYIIE|   5.0|-0.99232894|
|A1A4GJRXR9RAHJ|B000WFB83Q|   4.0| -1.7156762|
|A1ACU926NFOLZB|B00D3UVWQK|   5.0|   -3.09196|
|A1BBKX1M8KMGW6|B00BLCHL4Y|   4.0|-0.14413844|
| A1C43CL4E6K3E|B008LRB1O8|   5.0| -1.9676496|
|A1D0CBAZDWZRO6|B003ZHOWFY|   4.0| -1.5466051|
|A1D2C0WDCSHUWZ|B002TWWOHE|   4.0| -2.2607067|
|A1D2C0WDCSHUWZ|B003UPE4WC|   3.0|    0.59095|
|A1DUILQVFUW4